# Correlation Engine

Loads combined sentiment from `sentiment_outputs/` and stock prices,
then runs the full correlation analysis.

## Key differences from previous approach
- Uses all tickers with signal (500+) not just 24
- Finds key price movements FIRST, then looks for preceding sentiment
- Cross-ticker spillover with rolling windows
- Source attribution: which source triggered each signal

## Outputs saved to GitHub
| File | Contents |
|------|----------|
| `correlation_outputs/corr_matrix.csv` | Full correlation matrix |
| `correlation_outputs/key_moves.csv` | Big price moves + preceding sentiment |
| `correlation_outputs/best_per_ticker.csv` | Best signal per ticker |
| `correlation_outputs/spillover_pairs.csv` | Cross-ticker pairs |
| `correlation_outputs/source_attribution.csv` | Which source drives each ticker |

---
## 0. Install

In [1]:
# !pip install pandas requests python-dotenv scipy

---
## 1. Configuration & Load

In [2]:
import os, io, requests, warnings
import pandas as pd
import numpy as np
from scipy import stats
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv()

GITHUB_REPO      = 'annhmartin/dataviz-historical-stocks-AnnetteMartin'
GITHUB_TOKEN     = os.environ.get('GITHUB_TOKEN', None)
OUTPUT_PREFIX    = 'stock_tracking/sentiment_outputs'
CORR_PREFIX      = 'stock_tracking/correlation_outputs'
STOCKS_PREFIX    = 'stock_tracking/stocks'

ANALYSIS_START   = '2015-01-01'
ANALYSIS_END     = datetime.now(timezone.utc).strftime('%Y-%m-%d')
SENTIMENT_THRESHOLD = 0.05
ZSCORE_THRESHOLD    = 2.0
MIN_SIGNAL_DAYS     = 50    # raised from 10 — filters out micro-caps with tiny samples
MIN_CORR_N          = 200   # raised from 50 — needs 200 data points for meaningful correlation
MIN_PRICE           = 10.0  # raised from 5 — filters more micro-caps
MIN_COVERAGE_DAYS   = 100   # minimum signal days for spillover analysis
PRICE_HORIZONS      = [1, 2, 3, 5, 7, 10, 21]
SPILLOVER_WINDOW    = 90

def load_csv(path, token=None):
    url = f'https://raw.githubusercontent.com/{GITHUB_REPO}/main/{path}'
    headers = {'Authorization': f'Bearer {token}'} if token else {}
    resp = requests.get(url, headers=headers, timeout=60)
    if resp.status_code == 404: raise FileNotFoundError(path)
    resp.raise_for_status()
    content = resp.text.strip()
    if not content: return pd.DataFrame()
    return pd.read_csv(io.StringIO(content), low_memory=False)

def push_csv(df, path, token, msg=None):
    import base64
    GITHUB_API = 'https://api.github.com'
    if msg is None: msg = f'Update {path} - {len(df):,} rows'
    buf = io.StringIO()
    df.to_csv(buf, index=False)
    encoded = base64.b64encode(buf.getvalue().encode()).decode()
    url = f'{GITHUB_API}/repos/{GITHUB_REPO}/contents/{path}'
    headers = {'Authorization': f'Bearer {token}',
               'Accept': 'application/vnd.github+json',
               'X-GitHub-Api-Version': '2022-11-28'}
    check = requests.get(url, headers=headers, timeout=15)
    sha = check.json().get('sha') if check.status_code == 200 else None
    payload = {'message': msg, 'content': encoded}
    if sha: payload['sha'] = sha
    import time
    for attempt in range(3):
        resp = requests.put(url, headers=headers, json=payload, timeout=120)
        if resp.status_code in (200, 201):
            print(f'  Saved {path} ({len(df):,} rows)')
            return True
        if resp.status_code == 409 and attempt < 2:
            time.sleep(3)
            check = requests.get(url, headers=headers, timeout=15)
            sha = check.json().get('sha') if check.status_code == 200 else None
            if sha: payload['sha'] = sha
        else:
            print(f'  FAILED: {resp.status_code}')
            return False

print('Loading sentiment signals (quarterly files) ...')
sig_frames = []
for year in range(int(ANALYSIS_START[:4]), int(ANALYSIS_END[:4])+1):
    for q in [1, 2, 3, 4]:
        try:
            df_y = load_csv(f'{OUTPUT_PREFIX}/daily_signals_{year}_Q{q}.csv', GITHUB_TOKEN)
            sig_frames.append(df_y)
            print(f'  {year} Q{q}: {len(df_y):,} rows, {df_y["ticker"].nunique()} tickers')
        except FileNotFoundError:
            pass
# Fallback: also try yearly files (for compatibility)
if not sig_frames:
    for year in range(int(ANALYSIS_START[:4]), int(ANALYSIS_END[:4])+1):
        try:
            df_y = load_csv(f'{OUTPUT_PREFIX}/daily_signals_{year}.csv', GITHUB_TOKEN)
            sig_frames.append(df_y)
            print(f'  {year}: {len(df_y):,} rows (yearly file)')
        except FileNotFoundError:
            pass

if not sig_frames:
    raise RuntimeError('No sentiment data found. Run A_sentiment_engine.ipynb first.')

daily_signals = pd.concat(sig_frames, ignore_index=True)
daily_signals['date'] = pd.to_datetime(daily_signals['date'])
all_tickers = daily_signals['ticker'].unique().tolist()
print(f'\nLoaded: {len(daily_signals):,} ticker-days, {len(all_tickers):,} tickers')

Loading sentiment signals (quarterly files) ...
  2015 Q1: 175,680 rows, 1952 tickers
  2015 Q2: 177,632 rows, 1952 tickers
  2015 Q3: 179,584 rows, 1952 tickers
  2015 Q4: 179,584 rows, 1952 tickers
  2016 Q1: 177,632 rows, 1952 tickers
  2016 Q2: 177,632 rows, 1952 tickers
  2016 Q3: 179,584 rows, 1952 tickers
  2016 Q4: 179,584 rows, 1952 tickers
  2017 Q1: 175,680 rows, 1952 tickers
  2017 Q2: 177,632 rows, 1952 tickers
  2017 Q3: 179,584 rows, 1952 tickers
  2017 Q4: 179,584 rows, 1952 tickers
  2018 Q1: 175,680 rows, 1952 tickers
  2018 Q2: 177,632 rows, 1952 tickers
  2018 Q3: 179,584 rows, 1952 tickers
  2018 Q4: 179,584 rows, 1952 tickers
  2019 Q1: 175,680 rows, 1952 tickers
  2019 Q2: 177,632 rows, 1952 tickers
  2019 Q3: 179,584 rows, 1952 tickers
  2019 Q4: 179,584 rows, 1952 tickers
  2020 Q1: 177,632 rows, 1952 tickers
  2020 Q2: 177,632 rows, 1952 tickers
  2020 Q3: 179,584 rows, 1952 tickers
  2020 Q4: 179,584 rows, 1952 tickers
  2021 Q1: 175,680 rows, 1952 tickers
  

---
## 2. Load Stock Prices for All Tickers

In [3]:
print('Loading stock prices ...')
price_frames = []
missing = []

for i, ticker in enumerate(all_tickers):
    if i % 100 == 0: print(f'  {i}/{len(all_tickers)} ...', end='\r')
    try:
        df_p = load_csv(f'{STOCKS_PREFIX}/prices_{ticker}.csv', GITHUB_TOKEN)
        if df_p.empty or 'Close' not in df_p.columns:
            missing.append(ticker)
            continue
        df_p['Date']         = pd.to_datetime(df_p['Date'])
        df_p['ticker']       = ticker
        df_p['daily_return'] = df_p['Close'].pct_change(fill_method=None)
        # Filter out penny stocks — too noisy
        median_price = df_p['Close'].median()
        if median_price < MIN_PRICE:
            missing.append(ticker)
            continue
        df_p['roll_mean']    = df_p['daily_return'].rolling(60, min_periods=20).mean()
        df_p['roll_std']     = df_p['daily_return'].rolling(60, min_periods=20).std()
        df_p['zscore']       = ((df_p['daily_return'] - df_p['roll_mean']) / df_p['roll_std'])
        df_p['is_abnormal']  = df_p['zscore'].abs() >= ZSCORE_THRESHOLD
        price_frames.append(
            df_p[['Date','ticker','Close','daily_return','roll_std','zscore','is_abnormal']]
        )
    except FileNotFoundError:
        missing.append(ticker)

df_prices = pd.concat(price_frames, ignore_index=True) if price_frames else pd.DataFrame()
priced_tickers = df_prices['ticker'].unique().tolist() if not df_prices.empty else []
print(f'\nPrices loaded: {len(priced_tickers):,} tickers')
print(f'No price data: {len(missing):,} tickers')

Loading stock prices ...
  1900/1952 ...
Prices loaded: 1,513 tickers
No price data: 439 tickers


---
## 3. Find Key Price Movements First

Finds every date where a stock had an abnormal move (z-score > threshold),
then looks back 1-7 days to find what sentiment preceded it.
This is the correct order: price move → look back for sentiment.

In [4]:
print('Finding key price movements and preceding sentiment ...')

key_move_rows = []

sig_idx = daily_signals.set_index(['ticker','date'])

# Phantom ticker blocklist for key moves
KEY_MOVE_BLOCKLIST = {
    'RDIB','CAPS','TONX','GRRR','BULL','DFNS','WLDS','XFOR','GWAV',
    'CARE','CAR','SHIP','TWIN','HCSG','CENT','PPLI','APPS','PSIX',
    'FORM','HAPN','BPOPM','NSPR','GLAD','CBFV','LEDS','SHOE','WATT',
    'TAOP','TTMI','DFLI','FEAM','MASS','XOS','CDT','SERV','NTSK',
}

for ticker in priced_tickers:
    if ticker in KEY_MOVE_BLOCKLIST: continue
    price = df_prices[df_prices['ticker']==ticker].set_index('Date').sort_index()
    # Skip if median price too low
    if price['Close'].median() < MIN_PRICE: continue
    big_moves = price[price['zscore'].abs() >= ZSCORE_THRESHOLD]

    for move_date, mrow in big_moves.iterrows():
        # Look back up to 7 days for sentiment
        for lookback in range(1, 8):
            sent_date = move_date - pd.Timedelta(days=lookback)
            try:
                srow = sig_idx.loc[(ticker, sent_date)]
                # Prefer adaptive_sentiment, fall back to norm_sentiment
                sent_val = srow.get('adaptive_sentiment', np.nan)
                if pd.isna(sent_val):
                    sent_val = srow.get('norm_sentiment', np.nan)
                if pd.isna(sent_val): continue
                # Check if sentiment direction matched move direction
                move_dir  = 'up' if mrow['daily_return'] > 0 else 'down'
                sent_dir  = 'positive' if sent_val >= SENTIMENT_THRESHOLD else (
                            'negative' if sent_val <= -SENTIMENT_THRESHOLD else 'neutral')
                predicted = ((sent_dir=='positive' and move_dir=='up') or
                             (sent_dir=='negative' and move_dir=='down'))
                key_move_rows.append({
                    'ticker'         : ticker,
                    'move_date'      : move_date,
                    'return_pct'     : round(mrow['daily_return']*100, 3),
                    'zscore'         : round(mrow['zscore'], 3),
                    'move_direction' : move_dir,
                    'sent_date'      : sent_date,
                    'days_before'    : lookback,
                    'sentiment'      : round(sent_val, 4),
                    'sent_direction' : sent_dir,
                    'predicted'      : predicted,
                    'story_count'    : srow.get('story_count', 0),
                    'sources_active' : srow.get('sources_active', ''),
                })
                break  # use closest preceding sentiment day
            except KeyError:
                continue

df_key_moves = pd.DataFrame(key_move_rows)
if not df_key_moves.empty:
    df_key_moves['move_date'] = pd.to_datetime(df_key_moves['move_date'])
    df_key_moves['sent_date'] = pd.to_datetime(df_key_moves['sent_date'])
    total       = len(df_key_moves)
    predicted   = df_key_moves['predicted'].sum()
    no_sent     = len(df_prices[df_prices['is_abnormal']]) - total
    print(f'Key moves found    : {total:,}')
    print(f'Sentiment preceded : {total:,} ({total/(total+no_sent):.1%} of all big moves)')
    print(f'Correctly predicted: {predicted:,} ({predicted/total:.1%})')
    print(f'No sentiment found : {no_sent:,} (blind spots)')
    print('\nTop 10 biggest predicted moves:')
    print(df_key_moves[df_key_moves['predicted']]
          .sort_values('zscore', key=abs, ascending=False)
          .head(10)[['ticker','move_date','return_pct','zscore',
                     'days_before','sentiment','sources_active']]
          .to_string(index=False))

Finding key price movements and preceding sentiment ...
Key moves found    : 21,035
Sentiment preceded : 21,035 (9.8% of all big moves)
Correctly predicted: 5,061 (24.1%)
No sentiment found : 193,752 (blind spots)

Top 10 biggest predicted moves:
ticker  move_date  return_pct  zscore  days_before  sentiment        sources_active
    VS 2024-10-16     243.478   7.481            1     0.7150                     0
  DARE 2020-01-13      96.897   7.429            1     0.4440                     0
  GNPX 2022-01-03     167.176   7.395            1     0.9474 reddit_wallstreetbets
  FEED 2016-10-24      23.894   7.321            7     0.2668                     0
  JAGX 2019-03-18     192.453   7.312            1     1.1404                     0
    VS 2015-12-11     104.225   7.309            1     0.7150                     0
  VIVO 2019-06-24      96.172   7.280            1     0.5029                     0
   TGL 2024-08-20     141.379   7.275            4     0.6322                    

---
## 4. Full Correlation Matrix

In [5]:
print('Running correlation matrix (all tickers x all horizons) ...')
print('This may take 10-20 minutes for large universes.')

signal_cols = ['norm_sentiment','adaptive_sentiment','roll_3d','roll_5d',
               'roll_7d','roll_21d','sent_momentum','volume_surge']
signal_cols = [c for c in signal_cols
               if c in daily_signals.columns]

corr_results = []
tickers_with_both = [t for t in all_tickers if t in priced_tickers]
print(f'Running for {len(tickers_with_both):,} tickers with both signal and price data')

for i, ticker in enumerate(tickers_with_both):
    if i % 50 == 0: print(f'  {i}/{len(tickers_with_both)} ...', end='\r')
    sig   = daily_signals[daily_signals['ticker']==ticker].set_index('date').sort_index()
    price = df_prices[df_prices['ticker']==ticker].set_index('Date').sort_index()
    if len(sig) < MIN_SIGNAL_DAYS: continue

    for sig_col in signal_cols:
        if sig_col not in sig.columns: continue
        sent_series = sig[sig_col].dropna()
        if len(sent_series) < MIN_SIGNAL_DAYS: continue

        for horizon in PRICE_HORIZONS:
            pairs = []
            for date, sv in sent_series.items():
                future = price.index[price.index > date]
                if len(future) < horizon: continue
                ret = price.loc[future[horizon-1], 'daily_return']
                if not pd.isna(ret):
                    pairs.append((sv, ret))
            if len(pairs) < MIN_CORR_N: continue
            arr = np.array(pairs)
            # Skip if input is constant (no variance in sentiment)
            if np.std(arr[:,0]) < 0.001 or np.std(arr[:,1]) < 0.001: continue
            r, p = stats.pearsonr(arr[:,0], arr[:,1])
            corr_results.append({
                'ticker'     : ticker,
                'signal'     : sig_col,
                'horizon'    : horizon,
                'corr'       : round(r, 4),
                'pval'       : round(p, 4),
                'n'          : len(pairs),
                'significant': p < 0.05,
            })

df_corr = pd.DataFrame(corr_results)
print(f'\nCorrelation matrix: {len(df_corr):,} combinations')

best_per_ticker = (
    df_corr.assign(abs_corr=df_corr['corr'].abs())
    .sort_values('abs_corr', ascending=False)
    .drop_duplicates(subset='ticker')
    .drop(columns='abs_corr')
    .sort_values('corr', key=abs, ascending=False)
    .reset_index(drop=True)
)
print('\nTop 20 tickers by absolute correlation:')
print(best_per_ticker.head(20)[['ticker','signal','horizon','corr','pval','n','significant']]
      .to_string(index=False))

Running correlation matrix (all tickers x all horizons) ...
This may take 10-20 minutes for large universes.
Running for 1,513 tickers with both signal and price data
  1500/1513 ...
Correlation matrix: 24,139 combinations

Top 20 tickers by absolute correlation:
ticker             signal  horizon    corr   pval    n  significant
  FRSH adaptive_sentiment       21 -0.5343 0.0000  245         True
   VOR           roll_21d        2 -0.5033 0.0000  237         True
   GFS adaptive_sentiment        3  0.4161 0.0000  261         True
   ACT           roll_21d        5 -0.4059 0.0000  336         True
   IMA           roll_21d        2  0.3852 0.0000  211         True
  DNUT           roll_21d        3  0.3832 0.0000  312         True
  AMCI adaptive_sentiment       10  0.3694 0.0000  267         True
  DCTH           roll_21d        2  0.3667 0.0000  302         True
  ESTA adaptive_sentiment        2 -0.3519 0.0000  387         True
  PHAT           roll_21d        7  0.3167 0.0000  200  

---
## 5. Source Attribution

For each ticker, which source contributed the most to the signal?

In [6]:
print('Computing source attribution ...')

try:
    source_attr = load_csv(f'{OUTPUT_PREFIX}/source_attribution.csv', GITHUB_TOKEN)
    dominant = (
        source_attr.sort_values('item_count', ascending=False)
        .drop_duplicates(subset='ticker')
        .rename(columns={'source': 'dominant_source', 'item_count': 'dominant_count'})
    )
    print('Source breakdown across all tickers:')
    print(source_attr.groupby('source')['item_count'].sum()
          .sort_values(ascending=False).to_string())
    print('\nDominant source per ticker (sample):')
    print(dominant[['ticker','dominant_source','dominant_count']].head(20).to_string(index=False))
except FileNotFoundError:
    print('source_attribution.csv not found — run A_sentiment_engine.ipynb first')
    source_attr = pd.DataFrame()
    dominant    = pd.DataFrame()

Computing source attribution ...
Source breakdown across all tickers:
source
gdelt                      241536
hn                         131382
reddit_technology           43164
reddit_wallstreetbets       32316
reddit_stocks               26768
reddit_investing            23421
reddit_SecurityAnalysis      3747
edgar_8k                     3730
stocktwits                   2312

Dominant source per ticker (sample):
ticker dominant_source  dominant_count
  NVDA           gdelt           17000
  AAPL           gdelt           16250
  MSFT           gdelt           16010
  AMZN           gdelt           15500
 GOOGL           gdelt           15250
  CRWD           gdelt           14800
  COIN           gdelt           14272
  NFLX           gdelt           14250
  PYPL           gdelt           14000
   AMD           gdelt           14000
  PANW           gdelt           13712
  QCOM           gdelt           13505
  TSLA           gdelt           13500
  INTC           gdelt           

---
## 6. Cross-Ticker Spillover (Rolling 90-Day Windows)

In [7]:
print('Computing cross-ticker spillover pairs ...')
print('Note: with 500+ tickers this is O(n^2) — sampling top 50 by signal strength')

MIN_CONSISTENCY = 0.30
MIN_AVG_CORR    = 0.12
TOP_N_TICKERS   = 100  # top tickers by signal coverage

# Use well-covered tickers only for spillover — filters out micro-caps
coverage = (
    daily_signals.groupby('ticker')['norm_sentiment']
    .apply(lambda x: x.notna().sum())
)
# Only tickers with meaningful coverage AND in our priced universe
top_by_signal = (
    coverage[coverage >= MIN_COVERAGE_DAYS]
    .sort_values(ascending=False)
    .head(TOP_N_TICKERS)
    .index.tolist()
)
SPILL_BLOCKLIST = {
    'RDIB','CAPS','TONX','GRRR','BULL','DFNS','WLDS','XFOR','PPLI',
    'APPS','CARE','SHIP','TWIN','HCSG','CENT','PSIX','FORM','HAPN',
}
top_by_signal = [t for t in top_by_signal
                 if t in priced_tickers and t not in SPILL_BLOCKLIST]
print(f'Spillover universe: {len(top_by_signal)} tickers with >{MIN_COVERAGE_DAYS} signal days')
print(f'Running spillover for top {len(top_by_signal)} tickers by signal coverage')

sent_wide = daily_signals[daily_signals['ticker'].isin(top_by_signal)].pivot_table(
    index='date', columns='ticker', values='norm_sentiment'
)
fwd_wide_rows = {}
for ticker in top_by_signal:
    price = df_prices[df_prices['ticker']==ticker].set_index('Date').sort_index()
    fwd = price['Close'].pct_change(5, fill_method=None).shift(-5)
    fwd_wide_rows[ticker] = fwd
fwd_wide = pd.DataFrame(fwd_wide_rows)
fwd_wide.index = pd.to_datetime(fwd_wide.index)

spillover_rows = []
for st in top_by_signal:
    for pt in top_by_signal:
        if st == pt: continue
        if st not in sent_wide.columns or pt not in fwd_wide.columns: continue
        combined = pd.concat([
            sent_wide[st].rename('sent'),
            fwd_wide[pt].rename('ret')
        ], axis=1).dropna()
        if len(combined) < 60: continue
        roll = combined['sent'].rolling(SPILLOVER_WINDOW, min_periods=30).corr(combined['ret'])
        avg  = roll.mean()
        cons = max((roll > 0.2).mean(), (roll < -0.2).mean())
        if abs(avg) >= MIN_AVG_CORR and cons >= MIN_CONSISTENCY:
            spillover_rows.append({
                'sent_ticker' : st,
                'price_ticker': pt,
                'avg_corr'    : round(avg, 4),
                'consistency' : round(cons, 3),
                'direction'   : 'positive' if avg > 0 else 'negative',
            })

df_spillover = pd.DataFrame(spillover_rows).sort_values('avg_corr', key=abs, ascending=False)
print(f'Confirmed spillover pairs: {len(df_spillover)}')
if not df_spillover.empty:
    print(df_spillover.head(20).to_string(index=False))

Computing cross-ticker spillover pairs ...
Note: with 500+ tickers this is O(n^2) — sampling top 50 by signal strength
Spillover universe: 80 tickers with >100 signal days
Running spillover for top 80 tickers by signal coverage
Confirmed spillover pairs: 130
sent_ticker price_ticker  avg_corr  consistency direction
        RAY         ATGL    0.3765        0.716  positive
      GOOGL         PARK    0.3485        0.592  positive
       NFLX         PTRN   -0.3438        0.525  negative
       EDIT         IRON   -0.3187        0.710  negative
       ICON          PRE   -0.3142        0.648  negative
       SBUX         STEP    0.3041        0.694  positive
       IRON         GTLB   -0.3017        0.624  negative
       STRD         FIGR    0.3015        0.581  positive
       SHOP         ICON   -0.2898        0.623  negative
       BOOM         PTRN   -0.2787        0.525  negative
       SBUX         LYFT    0.2736        0.677  positive
       ICON         GTLB    0.2687        0.5

---
## 7. Save All Outputs

In [8]:
if GITHUB_TOKEN is None:
    print('GITHUB_TOKEN not set')
else:
    outputs = [
        (df_corr,        f'{CORR_PREFIX}/corr_matrix.csv',       'Full correlation matrix'),
        (best_per_ticker,f'{CORR_PREFIX}/best_per_ticker.csv',   'Best signal per ticker'),
        (df_key_moves,   f'{CORR_PREFIX}/key_moves.csv',         'Key moves with preceding sentiment'),
        (df_spillover,   f'{CORR_PREFIX}/spillover_pairs.csv',   'Cross-ticker spillover pairs'),
    ]
    if not source_attr.empty:
        outputs.append((source_attr, f'{CORR_PREFIX}/source_attribution.csv', 'Source attribution'))

    for df, path, desc in outputs:
        if df is not None and not df.empty:
            push_csv(df, path, GITHUB_TOKEN, desc)

    print('\nAll correlation outputs saved')

GITHUB_TOKEN not set


---
## 8. Source Weight Calibration

Measures how well each source's sentiment actually predicts the next price move,
then derives weights from that evidence rather than from intuition.

**Method**
1. For each ticker-day, work out which sources were active
2. Compare that day's sentiment direction against the forward return
3. A source's hit rate is the share of its signals that pointed the right way
4. Weight is scaled by how far the hit rate sits above random chance (50%)

A source at 50% is no better than a coin flip and gets a low weight no matter how
strong its sentiment scores look. Note that average absolute sentiment measures how
*polarized* a source is, not how *accurate* it is — the two are unrelated.

In [9]:
print('Calibrating source weights from predictive accuracy ...')

FORWARD_DAYS = 5      # horizon over which to judge a signal
MIN_SIGNALS  = 100    # a source needs this many signals to be scored

# daily_signals carries a sources_active column listing which sources fired that day
if 'sources_active' not in daily_signals.columns:
    print('sources_active column not found — re-run A_sentiment_engine first')
else:
    sig_col_cal = 'adaptive_sentiment' if 'adaptive_sentiment' in daily_signals.columns else 'norm_sentiment'

    # Forward return per ticker-day
    fwd_frames = []
    for ticker in priced_tickers:
        p = df_prices[df_prices['ticker'] == ticker].set_index('Date').sort_index()
        if len(p) < FORWARD_DAYS + 1:
            continue
        fwd = p['Close'].pct_change(FORWARD_DAYS, fill_method=None).shift(-FORWARD_DAYS)
        fwd_frames.append(pd.DataFrame({'ticker': ticker, 'date': p.index, 'fwd_ret': fwd.values}))
    df_fwd = pd.concat(fwd_frames, ignore_index=True) if fwd_frames else pd.DataFrame()

    if df_fwd.empty:
        print('No forward returns available')
    else:
        merged = daily_signals.merge(df_fwd, on=['ticker', 'date'], how='inner')
        merged = merged[merged[sig_col_cal].notna() & merged['fwd_ret'].notna()]
        merged = merged[merged[sig_col_cal].abs() >= SENTIMENT_THRESHOLD]
        print(f'Evaluating {len(merged):,} signal days against {FORWARD_DAYS}-day forward returns')

        # Explode the pipe-delimited source list so each row is one source-signal
        merged['sources_active'] = merged['sources_active'].fillna('').astype(str)
        exploded = merged.assign(source=merged['sources_active'].str.split('|')).explode('source')
        exploded['source'] = exploded['source'].str.strip()
        exploded = exploded[exploded['source'] != '']

        exploded['correct'] = (
            ((exploded[sig_col_cal] > 0) & (exploded['fwd_ret'] > 0)) |
            ((exploded[sig_col_cal] < 0) & (exploded['fwd_ret'] < 0))
        )

        stats_rows = []
        for source, grp in exploded.groupby('source'):
            n = len(grp)
            if n < MIN_SIGNALS:
                continue
            hit = float(grp['correct'].mean())
            # Binomial standard error, to show whether the edge is meaningful
            se = (hit * (1 - hit) / n) ** 0.5
            stats_rows.append({
                'source'      : source,
                'signals'     : n,
                'hit_rate'    : round(hit, 4),
                'edge_vs_coin': round(hit - 0.5, 4),
                'std_error'   : round(se, 4),
                'significant' : abs(hit - 0.5) > 2 * se,
                'avg_abs_sent': round(float(grp[sig_col_cal].abs().mean()), 4),
            })

        df_source_stats = pd.DataFrame(stats_rows).sort_values('hit_rate', ascending=False)

        if df_source_stats.empty:
            print(f'No source reached {MIN_SIGNALS} signals')
        else:
            # Weight scales with edge over chance. A source at 50% lands on the floor
            # weight regardless of how polarized its scores are.
            FLOOR, CEILING = 0.3, 1.6
            edges = df_source_stats['edge_vs_coin'].clip(lower=0)
            if edges.max() > 0:
                scaled = FLOOR + (edges / edges.max()) * (CEILING - FLOOR)
            else:
                scaled = pd.Series([1.0] * len(df_source_stats), index=df_source_stats.index)
            df_source_stats['suggested_weight'] = scaled.round(2)
            # A source with no statistically meaningful edge is held at the floor
            df_source_stats.loc[~df_source_stats['significant'], 'suggested_weight'] = FLOOR

            print()
            print(df_source_stats.to_string(index=False))
            print()
            print('Paste into SOURCE_WEIGHTS in A_sentiment_engine.ipynb:')
            print('SOURCE_WEIGHTS = {')
            for _, row in df_source_stats.iterrows():
                note = '' if row['significant'] else '   # not statistically significant\n'
                print(f"    {row['source']!r:28s}: {row['suggested_weight']},{note}")
            print('}')

            if GITHUB_TOKEN:
                push_csv(df_source_stats, f'{CORR_PREFIX}/source_weights.csv', GITHUB_TOKEN,
                         f'Source weight calibration: {len(df_source_stats)} sources')

Calibrating source weights from predictive accuracy ...
Evaluating 203,864 signal days against 5-day forward returns

                 source  signals  hit_rate  edge_vs_coin  std_error  significant  avg_abs_sent  suggested_weight
      reddit_technology     6988    0.5243        0.0243     0.0060         True        0.3173              1.60
  reddit_wallstreetbets     4883    0.5142        0.0142     0.0072        False        0.3073              0.30
          reddit_stocks     6192    0.5097        0.0097     0.0064        False        0.2987              0.30
                     hn    19488    0.5088        0.0088     0.0036         True        0.2803              0.77
       reddit_investing     5551    0.5084        0.0084     0.0067        False        0.3208              0.30
               edgar_8k      744    0.4973       -0.0027     0.0183        False        0.1569              0.30
reddit_SecurityAnalysis     1028    0.4942       -0.0058     0.0156        False        0.3

---
## 9. Phantom Ticker Audit

Text matching inevitably produces false positives: an article containing the word
"on" is not about ON Semiconductor. This section quantifies how much of the
matched universe is real and flags the entries most likely to be noise.

**Signals that a match is probably spurious**
1. The ticker has no entry in the SEC company register
2. The ticker is a common English word
3. Coverage is dominated by a single source, suggesting one text pattern is firing
4. Sentiment barely varies, which happens when the same phrase matches repeatedly

Output: `correlation_outputs/ticker_audit.csv`

In [10]:
print('Auditing matched tickers for false positives ...\n')

# SEC company register, the authoritative list of real tickers
sec_names = {}
try:
    resp = requests.get('https://www.sec.gov/files/company_tickers.json',
                        headers={'User-Agent': 'TechPulse research contact@example.com'},
                        timeout=30)
    if resp.status_code == 200:
        for entry in resp.json().values():
            t = str(entry.get('ticker', '')).upper().strip()
            n = str(entry.get('title', '')).strip()
            if t and n:
                sec_names[t] = n
    print(f'SEC register: {len(sec_names):,} companies')
except Exception as e:
    print(f'Could not reach SEC ({e}); the audit will skip the name check')

COMMON_WORDS = {
    'ON','CAN','HAS','ANY','ALL','NEW','NOW','ONE','TWO','WAY','DAY','MAN','OUT',
    'GET','GOT','PUT','SET','LET','RUN','HOW','WHY','WHO','TOP','BIG','APP','NET',
    'WEB','CAR','REAL','OPEN','HELP','FAST','LOVE','MOVE','NEWS','BOOK','EVER',
    'APPS','PDFS','FORM','CARE','SHIP','TWIN','WISH','SNAP','EDIT','MAPS','JOBS',
    'SHOP','CART','BOLT','GLOW','SWIM','BARK','IRON','WIRE','SEND','PICK','SEEK',
    'VIEW','FLOW','PIPE','BULK','RENT','DOCS','FILL','LINE','LINK','LIST','LOAD',
}

try:
    src_attr = load_csv(f'{OUTPUT_PREFIX}/source_attribution.csv', GITHUB_TOKEN)
except Exception:
    src_attr = pd.DataFrame()

coverage = (daily_signals.groupby('ticker')
            .agg(signal_days=('norm_sentiment', lambda x: int(x.notna().sum())),
                 total_stories=('story_count', 'sum'),
                 sent_std=('norm_sentiment', 'std'))
            .reset_index())
coverage = coverage[coverage['signal_days'] > 0]

audit_rows = []
for _, row in coverage.iterrows():
    t = str(row['ticker']).upper()
    flags = []

    if sec_names and t not in sec_names:
        flags.append('not in SEC register')
    if t in COMMON_WORDS:
        flags.append('common English word')
    if pd.notna(row['sent_std']) and row['sent_std'] < 0.01 and row['signal_days'] > 20:
        flags.append('sentiment barely varies')

    dominant_share, dominant_source = np.nan, ''
    if not src_attr.empty:
        sub = src_attr[src_attr['ticker'] == row['ticker']]
        if not sub.empty:
            total = sub['item_count'].sum()
            if total > 0:
                top = sub.loc[sub['item_count'].idxmax()]
                dominant_share  = float(top['item_count']) / float(total)
                dominant_source = str(top['source'])
                if dominant_share > 0.95 and row['signal_days'] > 50:
                    flags.append(f'{dominant_share:.0%} from {dominant_source} alone')

    audit_rows.append({
        'ticker'         : row['ticker'],
        'company'        : sec_names.get(t, ''),
        'signal_days'    : int(row['signal_days']),
        'total_stories'  : int(row['total_stories']) if pd.notna(row['total_stories']) else 0,
        'sent_std'       : round(float(row['sent_std']), 4) if pd.notna(row['sent_std']) else np.nan,
        'dominant_source': dominant_source,
        'dominant_share' : round(dominant_share, 3) if pd.notna(dominant_share) else np.nan,
        'flag_count'     : len(flags),
        'flags'          : '; '.join(flags),
        'likely_phantom' : len(flags) >= 1,
    })

df_audit = pd.DataFrame(audit_rows).sort_values(
    ['flag_count', 'signal_days'], ascending=[False, False])

total_t   = len(df_audit)
phantom_t = int(df_audit['likely_phantom'].sum())
clean_t   = total_t - phantom_t

print(f'\nTickers with signal : {total_t:,}')
print(f'Look legitimate     : {clean_t:,} ({clean_t/total_t:.1%})')
print(f'Flagged as suspect  : {phantom_t:,} ({phantom_t/total_t:.1%})')

if phantom_t:
    print('\nReason breakdown:')
    reasons = {}
    for f in df_audit[df_audit['likely_phantom']]['flags']:
        for part in str(f).split('; '):
            key = 'dominated by one source' if 'alone' in part else part
            reasons[key] = reasons.get(key, 0) + 1
    for k, v in sorted(reasons.items(), key=lambda x: -x[1]):
        print(f'  {v:5,}  {k}')

    print('\nMost heavily covered suspect tickers:')
    cols = ['ticker','company','signal_days','total_stories','flags']
    print(df_audit[df_audit['likely_phantom']].head(25)[cols].to_string(index=False))

print('\nMost heavily covered tickers that look legitimate:')
cols2 = ['ticker','company','signal_days','total_stories','dominant_source']
print(df_audit[~df_audit['likely_phantom']].nlargest(20, 'signal_days')[cols2].to_string(index=False))

if GITHUB_TOKEN:
    push_csv(df_audit, f'{CORR_PREFIX}/ticker_audit.csv', GITHUB_TOKEN,
             f'Ticker audit: {phantom_t:,} of {total_t:,} flagged')

    suspect = df_audit[df_audit['likely_phantom']]['ticker'].tolist()
    print(f'\nTo exclude these from analysis, add to the blocklist in A_sentiment_engine:')
    print('WORD_BLOCKLIST.update({')
    for i in range(0, min(len(suspect), 60), 10):
        chunk = suspect[i:i+10]
        print('    ' + ', '.join(repr(x) for x in chunk) + ',')
    if len(suspect) > 60:
        print(f'    # ... and {len(suspect)-60:,} more, see ticker_audit.csv')
    print('})')

Auditing matched tickers for false positives ...

SEC register: 10,411 companies

Tickers with signal : 1,952
Look legitimate     : 1,910 (97.8%)
Flagged as suspect  : 42 (2.2%)

Reason breakdown:
     29  common English word
      7  dominated by one source
      6  not in SEC register

Most heavily covered suspect tickers:
ticker                             company  signal_days  total_stories                                flags
  APPS               Digital Turbine, Inc.         2231           5674                  common English word
  EVER                     EverQuote, Inc.         2156           4468                  common English word
   CAR             AVIS BUDGET GROUP, INC.         1772           3593                  common English word
  CRWD          CrowdStrike Holdings, Inc.         1543          15463                 95% from gdelt alone
  PANW              Palo Alto Networks Inc         1350          13988                 97% from gdelt alone
  DDOG                   